This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

### Instalación de dependencias
En el siguiente bloque de código, nos encargamos de instalar y actualizar las librerías necesarias para nuestro modelo de Deep Learning.

**Funciones y parámetros:**
* `!pip install`: Es el comando estándar de Python para instalar paquetes desde el Python Package Index (PyPI). El signo de exclamación `!` permite ejecutar comandos de terminal directamente desde una celda de Colab.
* `keras`: La librería principal de Deep Learning que usaremos para construir y entrenar modelos.
* `keras-hub`: Una extensión de Keras que proporciona modelos preentrenados y componentes útiles para tareas específicas como NLP o Visión Computacional.
* `--upgrade`: Este parámetro asegura que, si la librería ya está instalada, se actualice a su versión más reciente. Es crucial para evitar incompatibilidades con funciones nuevas.
* `-q` (quiet): Este parámetro silencia la salida de texto de la instalación, mostrando solo errores o advertencias, lo que mantiene el cuaderno limpio de registros de instalación extensos.

In [ ]:
!pip install keras keras-hub --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.0/230.0 kB 16.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.32.0 which is incompatible.


### Configuración del Backend de Keras
Keras 3 permite funcionar sobre diferentes motores matemáticos (backends) como TensorFlow, PyTorch o JAX. En el siguiente bloque elegimos JAX.

**Funciones y parámetros:**
* `import os`: Importa el módulo nativo de Python que permite interactuar con el sistema operativo y modificar variables de entorno.
* `os.environ["KERAS_BACKEND"] = "jax"`: `os.environ` es un diccionario que contiene las variables de entorno del sistema. Al asignar `"jax"` a la clave `"KERAS_BACKEND"`, le indicamos a Keras que utilice JAX como su motor de ejecución matemática. JAX es conocido por su alta velocidad y optimización en compilación (XLA) para GPUs y TPUs.

In [2]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [3]:
# Desactivar la pre-reserva completa de memoria de JAX para evitar errores de cuSolver
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [4]:
import jax
import os

# Verificar los dispositivos disponibles para JAX de forma correcta
try:
    backend = jax.default_backend()
except AttributeError:
    backend = "No se pudo detectar"

devices = jax.devices()
print("Backend activo de JAX:", backend)
print("Dispositivos detectados:", devices)

# Validar que la variable de entorno se cargó correctamente en el proceso actual
print("XLA_PYTHON_CLIENT_PREALLOCATE actual:", os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"))

Backend activo de JAX: gpu
Dispositivos detectados: [CudaDevice(id=0)]
XLA_PYTHON_CLIENT_PREALLOCATE actual: false


In [5]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## Language models and the Transformer

### The language model

#### Training a Shakespeare language model

### Descarga del conjunto de datos (Dataset)
En el siguiente bloque, descargaremos un archivo de texto con las obras de Shakespeare, el cual usaremos para entrenar un modelo generativo de lenguaje.

**Funciones y parámetros:**
* `keras.utils.get_file()`: Una función muy útil de Keras para descargar archivos desde una URL y guardarlos en el caché local del sistema.
* `origin`: Parámetro que recibe la URL directa (string) de donde se descargará el archivo. En este caso, apunta a un archivo de texto en los servidores de Google.
* `open(filename, "r").read()`:
    * `open()`: Función nativa de Python para abrir archivos.
    * `"r"`: El parámetro de modo que indica "read" (solo lectura).
    * `.read()`: Método que lee todo el contenido del archivo de texto y lo carga en la variable `shakespeare` como un gran string (cadena de texto).

In [ ]:
import keras

filename = keras.utils.get_file(
    origin=(
        "https://storage.googleapis.com/download.tensorflow.org/"
        "data/shakespeare.txt"
    ),
)
shakespeare = open(filename, "r").read()

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


In [ ]:
shakespeare[:250]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n'

### Creación de Secuencias y TensorFlow Dataset
En este bloque tomamos el texto completo y lo dividimos en secuencias pequeñas para que el modelo pueda aprender a predecir el siguiente carácter basándose en los anteriores.

**Funciones y parámetros:**
*   `sequence_length = 100`: Define la longitud de cada fragmento de texto (100 caracteres).
*   `split_input(...)`: Una función generadora de Python (usando `yield`) que recorre el texto y extrae porciones de tamaño `sequence_length`.
*   `features` y `labels`: `features` contiene el texto de entrada (sin el último carácter) y `labels` contiene el texto objetivo, desplazado un carácter hacia adelante. Esto enseña al modelo que dado el carácter $i$, debe predecir el carácter $i+1$.
*   `tf.data.Dataset.from_tensor_slices(...)`: Crea un objeto `Dataset` altamente optimizado de TensorFlow. Empaqueta las listas `features` y `labels` en un formato eficiente para alimentar la tarjeta gráfica (GPU/TPU) durante el entrenamiento.

In [ ]:
import tensorflow as tf

sequence_length = 100

def split_input(input, sequence_length):
    for i in range(0, len(input), sequence_length):
        yield input[i : i + sequence_length]

features = list(split_input(shakespeare[:-1], sequence_length))
labels = list(split_input(shakespeare[1:], sequence_length))
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

In [ ]:
x, y = next(dataset.as_numpy_iterator())
x[:50], y[:50]

(b'First Citizen:\nBefore we proceed any further, hear',
 b'irst Citizen:\nBefore we proceed any further, hear ')

### Vectorización de Texto (Tokenización)
Los modelos matemáticos no entienden letras, solo números. En este bloque convertimos los caracteres en números enteros (IDs).

**Funciones y parámetros:**
*   `layers.TextVectorization(...)`: Capa de preprocesamiento de Keras que transforma texto en secuencias de enteros.
    *   `standardize=None`: Evita que Keras modifique el texto (como pasar a minúsculas o borrar puntuación), ya que para Shakespeare queremos mantener mayúsculas y signos.
    *   `split="character"`: Le indica a la capa que divida el texto letra por letra (carácter por carácter) en lugar de palabra por palabra.
    *   `output_sequence_length=sequence_length`: Fija el tamaño de salida para que todas las secuencias midan exactamente 100.
*   `.adapt(...)`: Analiza el dataset (en este caso usando un `.map` para ver solo el texto, ignorando los labels) y crea el vocabulario. Asigna un número único a cada carácter único que encuentra.

In [ ]:
from keras import layers

tokenizer = layers.TextVectorization(
    standardize=None,
    split="character",
    output_sequence_length=sequence_length,
)
tokenizer.adapt(dataset.map(lambda text, labels: text))

In [ ]:
vocabulary_size = tokenizer.vocabulary_size()
vocabulary_size

67

### Preparación del Dataset para Entrenamiento
Aquí aplicamos la vectorización al dataset y lo preparamos para el entrenamiento mediante mezcla por lotes.

**Funciones y parámetros:**
*   `.map(...)`: Aplica una función a cada elemento del dataset. Aquí usamos una función `lambda` (función anónima) para pasar tanto las características (`features`) como las etiquetas (`labels`) por el `tokenizer`.
*   `num_parallel_calls=8`: Permite que la transformación de datos se ejecute en paralelo usando 8 hilos del procesador, acelerando el proceso.
*   `.shuffle(10_000)`: Mezcla los datos aleatoriamente utilizando un búfer de 10,000 elementos. Evita que el modelo memorice el orden del libro.
*   `.batch(64)`: Agrupa los datos en paquetes (lotes) de 64 secuencias. Entrenar por lotes hace que el proceso sea mucho más estable y rápido en las GPUs.
*   `.cache()`: Guarda el dataset procesado en la memoria RAM o disco después de la primera época, por lo que las siguientes épocas leerán los datos instantáneamente.

In [ ]:
dataset = dataset.map(
    lambda features, labels: (tokenizer(features), tokenizer(labels)),
    num_parallel_calls=8,
)
training_data = dataset.shuffle(10_000).batch(64).cache()

### Construcción del Modelo de Lenguaje (RNN)
A continuación creamos la arquitectura neuronal del modelo generativo usando la API Funcional de Keras.

**Funciones y parámetros:**
*   `layers.Input(...)`: Define el tensor de entrada (forma y tipo de datos).
*   `layers.Embedding(vocabulary_size, embedding_dim)`: Convierte los números enteros (IDs de caracteres) en vectores densos de 256 dimensiones (`embedding_dim`). Ayuda al modelo a entender la "similitud" entre distintos caracteres.
*   `layers.GRU(hidden_dim, return_sequences=True)`: Una capa Recurrente Gated Recurrent Unit (GRU) con 1024 neuronas (`hidden_dim`). Es excelente para procesar secuencias porque tiene "memoria". `return_sequences=True` indica que debe devolver una predicción para *cada* paso de la secuencia (100 predicciones), no solo al final.
*   `layers.Dropout(0.1)`: Apaga aleatoriamente el 10% de las neuronas durante el entrenamiento. Es una técnica de regularización que evita el sobreajuste (que el modelo memorice en lugar de aprender).
*   `layers.Dense(vocabulary_size, activation="softmax")`: La capa final que clasifica. Emite probabilidades (`softmax`) sobre cuál de los 67 caracteres del vocabulario es el siguiente.
*   `keras.Model(...)`: Ensambla el modelo indicando cuáles son sus entradas y sus salidas.

In [ ]:
embedding_dim = 256
hidden_dim = 1024

inputs = layers.Input(shape=(sequence_length,), dtype="int", name="token_ids")
x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x = layers.GRU(hidden_dim, return_sequences=True)(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)

In [ ]:
model.summary(line_length=80)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)            │ (None, 100)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ embedding (Embedding)             │ (None, 100, 256)         │        17,152 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ gru (GRU)                         │ (None, 100, 1024)        │     3,938,304 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout (Dropout)                 │ (None, 100, 1024)        │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense (Dense)                     │ (None, 100, 67)          │        68,675 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

 Total params: 4,024,131 (15.35 MB)

 Trainable params: 4,024,131 (15.35 MB)

 Non-trainable params: 0 (0.00 B)

### Compilación y Entrenamiento del Modelo
Finalmente, preparamos el modelo para aprender y lo entrenamos sobre los datos.

**Funciones y parámetros:**
*   `.compile(...)`: Configura el proceso de aprendizaje.
    *   `optimizer="adam"`: El algoritmo que actualiza los pesos de la red. Adam es muy eficiente y adaptativo.
    *   `loss="sparse_categorical_crossentropy"`: La función de pérdida. Penaliza al modelo cuando se equivoca. Se usa "sparse" porque nuestras etiquetas son números enteros (IDs de caracteres), no vectores one-hot.
    *   `metrics=["sparse_categorical_accuracy"]`: Métrica para monitorear qué porcentaje de veces el modelo acierta el siguiente carácter exacto.
*   `.fit(training_data, epochs=5)`: Inicia el ciclo de entrenamiento. Pasará por todo el dataset de Shakespeare 5 veces (`epochs=5`). Cada pasada ajusta los pesos para reducir la pérdida.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
model.fit(training_data, epochs=5)

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 20s 89ms/step - loss: 2.7190 - sparse_categorical_accuracy: 0.2764
Epoch 2/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 14s 78ms/step - loss: 1.9950 - sparse_categorical_accuracy: 0.4162
Epoch 3/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 14s 83ms/step - loss: 1.7283 - sparse_categorical_accuracy: 0.4869
Epoch 4/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 15s 87ms/step - loss: 1.5773 - sparse_categorical_accuracy: 0.5268
Epoch 5/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 17s 95ms/step - loss: 1.4843 - sparse_categorical_accuracy: 0.5510


#### Generating Shakespeare

### Creación del Modelo de Inferencia (Generación)
Para generar texto carácter por carácter, necesitamos un modelo ligeramente diferente al de entrenamiento. Este modelo debe ser capaz de recordar su estado interno paso a paso.

**Funciones y parámetros:**
*   `keras.Input(shape=(1,), ...)`: La entrada ahora es un solo carácter a la vez (shape de 1), no una secuencia de 100.
*   `input_state = keras.Input(shape=(hidden_dim,), ...)`: Una nueva entrada que representa la "memoria" del modelo (el estado oculto de la GRU).
*   `return_state=True`: Al llamar a la capa GRU, le pedimos que además de la predicción, nos devuelva su estado interno actualizado (`output_state`).
*   `generation_model.set_weights(model.get_weights())`: ¡Paso crucial! Copiamos los pesos (lo que aprendió) del modelo que entrenamos antes a este nuevo modelo generador.

In [ ]:
inputs = keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state = keras.Input(shape=(hidden_dim,), name="state")

x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x, output_state = layers.GRU(hidden_dim, return_state=True)(
    x, initial_state=input_state
)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
generation_model = keras.Model(
    inputs=(inputs, input_state),
    outputs=(outputs, output_state),
)
generation_model.set_weights(model.get_weights())

### Preparación de Diccionarios y Texto Inicial (Prompt)
Antes de generar, necesitamos formas rápidas de convertir letras a números y viceversa, además de darle un punto de partida al modelo.

**Funciones y parámetros:**
*   `tokenizer.get_vocabulary()`: Extrae la lista de todos los caracteres que el tokenizador conoce.
*   `dict(zip(...))`: Crea diccionarios de Python. `char_to_id` nos permite buscar un carácter y obtener su número, mientras que `id_to_char` hace lo contrario (necesario para leer lo que predice el modelo).
*   `prompt`: Es el texto

In [ ]:
tokens = tokenizer.get_vocabulary()
token_ids = range(vocabulary_size)
char_to_id = dict(zip(tokens, token_ids))
id_to_char = dict(zip(token_ids, tokens))

prompt = """
KING RICHARD III:
"""

### Procesamiento del Prompt y Estado Inicial
Leemos el prompt inicial para que la red neuronal construya su memoria (estado) antes de empezar a inventar texto.

**Funciones y parámetros:**
*   `keras.ops.zeros(shape=(1, hidden_dim))`: Crea un vector de ceros que representa la memoria vacía inicial de la red.
*   `keras.ops.expand_dims([token_id], axis=0)`: Añade una dimensión extra al ID del carácter para convertirlo en un "lote de 1" (formato que exige Keras).
*   `generation_model.predict((inputs, state), verbose=0)`: Pasa el carácter y el estado actual por la red. Obtenemos una predicción y el nuevo estado actualizado. `verbose=0` silencia la barra de progreso para que no imprima cientos de líneas.

In [ ]:
input_ids = [char_to_id[c] for c in prompt]
state = keras.ops.zeros(shape=(1, hidden_dim))
for token_id in input_ids:
    inputs = keras.ops.expand_dims([token_id], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

### Bucle de Generación de Caracteres
Aquí es donde ocurre la magia: usamos las predicciones pasadas para generar el futuro texto.

**Funciones y parámetros:**
*   `np.argmax(predictions, axis=-1)`: La red devuelve 67 probabilidades (una por cada posible siguiente carácter). `argmax` elige el índice del carácter con la probabilidad más alta (la predicción más segura).
*   Bucle `for`: Repite el proceso `max_length` veces (250 caracteres). Cada vez, toma el carácter recién generado (`next_char`) y lo usa como `inputs` para el siguiente paso, pasando también el último `state`.

In [ ]:
import numpy as np

generated_ids = []
max_length = 250
for i in range(max_length):
    next_char = int(np.argmax(predictions, axis=-1)[0])
    generated_ids.append(next_char)
    inputs = keras.ops.expand_dims([next_char], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

In [ ]:
output = "".join([id_to_char[token_id] for token_id in generated_ids])
print(prompt + output)


KING RICHARD III:
And what shall be so the sen that shall be so the sea
That shall be so the state of the sease to the sea
That shall be so the state of the sease to the sea
That shall be so the state of the sease to the sea
That shall be so the state of the sease to 


### Sequence-to-sequence learning

#### English-to-Spanish translation

### Descarga del Dataset de Traducción (Seq2Seq)
Comenzamos un nuevo proyecto: traducir de inglés a español usando un modelo Secuencia-a-Secuencia.

**Funciones y parámetros:**
*   `keras.utils.get_file(..., extract=True)`: Descarga un archivo ZIP que contiene pares de oraciones en inglés y español. El parámetro `extract=True` descomprime automáticamente el archivo ZIP después de descargarlo.
*   `pathlib.Path(...)`: Una librería nativa de Python elegante para manejar rutas de archivos de manera compatible en Windows, Mac o Linux. Navega hacia la ruta del archivo de texto extraído `spa.txt`.

In [ ]:
import pathlib

zip_path = keras.utils.get_file(
    origin=(
        "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    ),
    fname="spa-eng",
    extract=True,
)
text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


### Preparación de Pares de Traducción y Tokens Especiales
Leemos el archivo y preparamos las oraciones agregando marcas especiales que guiarán al modelo decodificador.

**Funciones y parámetros:**
*   `.split("\n")`: Divide el texto masivo en una lista, donde cada elemento es una línea del archivo.
*   `line.split("\t")`: Divide cada línea usando el tabulador (`\t`), separando la oración en inglés de la española.
*   `"[start] " + spanish + " [end]"`: Se añaden tokens especiales al español (el idioma a generar). `[start]` le avisa al modelo cuándo empezar a traducir, y `[end]` le indica al modelo cuándo ha terminado la oración.

In [ ]:
with open(text_path) as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    spanish = "[start] " + spanish + " [end]"
    text_pairs.append((english, spanish))

In [ ]:
import random
random.choice(text_pairs)

('The Italians often drink coffee.',
 '[start] Los italianos a menudo toman café. [end]')

### División del Conjunto de Datos (Splitting)
Para saber si nuestro modelo realmente está aprendiendo a traducir y no solo memorizando las oraciones, dividimos nuestras parejas de texto en tres grupos: entrenamiento (train), validación (val) y prueba (test).

**Funciones y parámetros:**
*   `random.shuffle(text_pairs)`: Desordena aleatoriamente todas las parejas de oraciones para que no haya un sesgo (por ejemplo, que todas las oraciones cortas estén al principio).
*   `val_samples = int(0.15 * len(text_pairs))`: Calcula el 15% del total de datos para usarlo como validación (para evaluar mientras se entrena).
*   `train_pairs`, `val_pairs`, `test_pairs`: Se usan rebanadas (slices) de listas en Python (`[:]`) para separar los datos correspondientes. Al entrenamiento le asignamos el 70% restante.

In [ ]:
import random

random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

### Tokenización y Estandarización de Textos
En este bloque configuramos cómo se procesará el texto de ambos idiomas. El español requiere un trato especial por sus signos de puntuación únicos (como `¿`).

**Funciones y parámetros:**
*   `custom_standardization(...)`: Una función personalizada que primero pasa todo el texto a minúsculas (`tf.strings.lower`) y luego usa una expresión regular (`tf.strings.regex_replace`) para borrar la puntuación, incluyendo los signos de interrogación de apertura `¿`.
*   `vocab_size = 15000` y `sequence_length = 20`: Limitamos el vocabulario a las 15,000 palabras más comunes y la longitud máxima de oración a 20 palabras.
*   `english_tokenizer` y `spanish_tokenizer`: Instancias de `layers.TextVectorization`. Nota que al español le pasamos `standardize=custom_standardization` y una longitud de secuencia de 21 (`sequence_length + 1`) porque le agregamos los tokens `[start]` y `[end]`.
*   `.adapt(...)`: Los tokenizadores analizan los textos de entrenamiento y construyen sus vocabularios respectivos (uno para inglés y otro para español).

In [ ]:
import string
import re

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

vocab_size = 15000
sequence_length = 20

english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

### Empaquetado de Datos para Seq2Seq
Para que la red neuronal procese la traducción eficientemente, preparamos un `tf.data.Dataset` especial. El modelo recibirá como entrada el texto en inglés y el texto en español desfasado por una palabra.

**Funciones y parámetros:**
*   `format_dataset(...)`: Configura la estructura `(inputs, targets)`. `inputs` incluye el inglés completo y el español sin el último token. `targets` (lo que debe predecir) es el español sin el primer token. Esto obliga al modelo a predecir la siguiente palabra basándose en las anteriores.
*   `sample_weights = labels != 0`: Crea una máscara para ignorar los ceros (padding) en el cálculo del error (loss). Así el modelo no "aprende" simplemente a predecir espacios en blanco.
*   `make_dataset(...)`: Junta las características y las empaqueta en lotes (`.batch(batch_size)`), las mezcla (`.shuffle`) y las guarda en memoria caché (`.cache()`) para mayor velocidad.

In [ ]:
batch_size = 64

# Pre-tokenizamos los textos utilizando directamente los tokenizadores fuera de tf.data
# para evitar conflictos entre el grafo de TensorFlow y el backend de JAX
train_eng_tokenized = english_tokenizer(train_english_texts)
train_spa_tokenized = spanish_tokenizer(train_spanish_texts)

val_eng_texts = [pair[0] for pair in val_pairs]
val_spa_texts = [pair[1] for pair in val_pairs]
val_eng_tokenized = english_tokenizer(val_eng_texts)
val_spa_tokenized = spanish_tokenizer(val_spa_texts)

def format_dataset(eng_tokens, spa_tokens):
    features = {
        "english": eng_tokens,
        "spanish": spa_tokens[:, :-1]
    }
    labels = spa_tokens[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(eng_tokens, spa_tokens):
    dataset = tf.data.Dataset.from_tensor_slices((eng_tokens, spa_tokens))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_eng_tokenized, train_spa_tokenized)
val_ds = make_dataset(val_eng_tokenized, val_spa_tokenized)

In [ ]:
inputs, targets, sample_weights = next(iter(train_ds))
print(inputs["english"].shape)

(64, 20)


In [ ]:
print(inputs["spanish"].shape)

(64, 20)


In [ ]:
print(targets.shape)

(64, 20)


In [ ]:
print(sample_weights.shape)

(64, 20)


#### Sequence-to-sequence learning with RNNs

### El Codificador (Encoder)
Esta es la primera mitad del modelo de traducción. Su trabajo es "leer" la frase en inglés y comprimirla en un solo vector numérico que representa su significado completo.

**Funciones y parámetros:**
*   `source = keras.Input(...)`: Define el punto de entrada para el texto en inglés.
*   `layers.Embedding(...)`: Convierte los números (IDs de palabras en inglés) en vectores con significado.
*   `layers.Bidirectional(..., merge_mode="sum")`: ¡El truco principal del codificador! Esta capa lee la frase en inglés de izquierda a derecha y luego de derecha a izquierda simultáneamente. Esto ayuda a comprender mejor el contexto de cada palabra. `merge_mode="sum"` suma los resultados de ambas lecturas.
*   `encoder_output`: Este resultado final contiene el "resumen" de toda la oración en inglés.

In [ ]:
embed_dim = 256
hidden_dim = 1024

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, embed_dim)(source)
rnn_layer = layers.GRU(hidden_dim)
rnn_layer = layers.Bidirectional(rnn_layer, merge_mode="sum")
encoder_output = rnn_layer(x)

### El Decodificador (Decoder)
Esta es la segunda mitad del modelo. Toma el "resumen" generado por el codificador y lo utiliza para empezar a generar la frase en español, palabra por palabra.

**Funciones y parámetros:**
*   `target = keras.Input(...)`: Define el punto de entrada para el texto parcial en español.
*   `x = rnn_layer(x, initial_state=encoder_output)`: Aquí se conectan las dos mitades. Le pasamos el `encoder_output` como el estado inicial (`initial_state`) de la capa recurrente del decodificador. De esta forma, el decodificador "sabe" qué es lo que tiene que traducir.
*   `target_predictions = layers.Dense(vocab_size, activation="softmax")(x)`: La capa final analiza las salidas de la RNN y emite probabilidades (`softmax`) sobre cuál de las 15,000 palabras del vocabulario en español es la correcta para la siguiente posición.

In [ ]:
target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, embed_dim)(target)
rnn_layer = layers.GRU(hidden_dim, return_sequences=True)
x = rnn_layer(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
seq2seq_rnn = keras.Model([source, target], target_predictions)

### Resumen de la Arquitectura del Modelo
En este breve bloque visualizamos la estructura final del modelo ensamblado.

**Funciones y parámetros:**
*   `seq2seq_rnn.summary(line_length=80)`: Imprime una tabla detallada con todas las capas del modelo, la forma de sus salidas (shape) y el número de parámetros (pesos) que aprenderá la red. `line_length=80` simplemente ajusta el ancho de la impresión para que sea más legible.

In [ ]:
seq2seq_rnn.summary(line_length=80)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_2           │ (None, None, 256) │   3,840,000 │ english[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_3           │ (None, None, 256) │   3,840,000 │ spanish[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ bidirectional         │ (None, 1024)      │   7,876,608 │ embedding_2[0][0]  │
│ (Bidirectional)       │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ gru_3 (GRU)           │ (None, None,      │   3,938,304 │ embedding_3[0][0], │
│                       │ 1024)             │             │ bidirectional[0][… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_1 (Dropout)   │ (None, None,      │           0 │ gru_3[0][0]        │
│                       │ 1024)             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense_2 (Dense)       │ (None, None,      │  15,375,000 │ dropout_1[0][0]    │
│                       │ 15000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 34,869,912 (133.02 MB)

 Trainable params: 34,869,912 (133.02 MB)

 Non-trainable params: 0 (0.00 B)

### Compilación y Entrenamiento del Modelo Seq2Seq
Preparamos nuestro modelo para aprender la tarea de traducción y lanzamos el ciclo de entrenamiento.

**Funciones y parámetros:**
*   `.compile(...)`:
    *   `optimizer="adam"`: Optimizador estándar y eficiente para ajustar los pesos.
    *   `loss="sparse_categorical_crossentropy"`: Calcula el error entre la palabra que el modelo predijo y la palabra real en español.
    *   `weighted_metrics=["accuracy"]`: Usamos `weighted_metrics` en lugar de `metrics` simples porque previamente configuramos `sample_weights` en nuestro dataset para ignorar el padding (los espacios vacíos). Esto nos da una precisión real de las palabras predichas.
*   `.fit(train_ds, epochs=5, validation_data=val_ds)`: Entrena el modelo durante 5 épocas usando nuestro dataset de entrenamiento (`train_ds`). Además, al final de cada época, evalúa su rendimiento en datos que nunca ha visto (`val_ds`), lo que nos ayuda a verificar que no esté sobreajustando (memorizando).

In [ ]:
seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
seq2seq_rnn.fit(train_ds, epochs=5, validation_data=val_ds)

Epoch 1/5
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 118s 87ms/step - accuracy: 0.2800 - loss: 1.6780 - val_accuracy: 0.3823 - val_loss: 1.2764
Epoch 2/5
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 119s 92ms/step - accuracy: 0.4126 - loss: 1.2032 - val_accuracy: 0.4665 - val_loss: 1.0304
Epoch 3/5
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 119s 91ms/step - accuracy: 0.4901 - loss: 0.9505 - val_accuracy: 0.5237 - val_loss: 0.8890
Epoch 4/5
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 119s 91ms/step - accuracy: 0.5515 - loss: 0.7667 - val_accuracy: 0.5568 - val_loss: 0.8174
Epoch 5/5
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 119s 92ms/step - accuracy: 0.6055 - loss: 0.6288 - val_accuracy: 0.5777 - val_loss: 0.7782


### Generación de Traducciones (Inferencia)
Una vez entrenado, creamos una función para probar el modelo con oraciones nuevas. A diferencia del entrenamiento donde le damos las respuestas correctas, aquí el modelo debe predecir paso a paso.

**Funciones y parámetros:**
*   `spa_index_lookup`: Un diccionario inverso que convierte los números (IDs) que predice la red de vuelta a palabras legibles en español.
*   `decoded_sentence = "[start]"`: Inicializa la oración en español con el token de inicio.
*   `seq2seq_rnn.predict(inputs, verbose=0)`: Pide al modelo que prediga la siguiente palabra. Le pasamos tanto la frase en inglés (constante) como la frase parcial en español que hemos construido hasta ahora.
*   `np.argmax(...)`: Elige la palabra con mayor probabilidad dentro del vocabulario de 15,000 palabras.
*   `if sampled_token == "[end]": break`: El bucle se detiene cuando el modelo decide que la oración ha terminado y genera el token de cierre.
*   El último bucle `for` selecciona aleatoriamente 5 oraciones del conjunto de prueba y muestra su traducción generada.

In [ ]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = seq2seq_rnn.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
How am I even going to buy this?
[start] cómo puedo comprar esta frase [end]
-
I learned it in school.
[start] lo aprendí en la escuela [end]
-
As soon as he went out, it began to rain.
[start] tan pronto como llegues comenzó a llover [end]
-
Write these words in your notebook.
[start] escriba tu palabra en el menú [end]
-
Tom is unable to work this week.
[start] tom no es capaz de trabajar esta noche [end]


# Taller 6

## Punto 1
### Entrenamiento con libro en español -> Don quijote

### 1.1 Descarga y Preparación del Dataset (Don Quijote)
En esta sección descargamos el libro "Don Quijote" desde Project Gutenberg. A diferencia del ejemplo de Shakespeare, aquí trabajamos con texto en español.

**Cambios respecto al ejemplo original:**
* Se utiliza `encoding="utf-8-sig"` al leer el archivo para asegurar que las tildes y la letra "ñ" se mantengan intactas.
* Se recorta el texto para eliminar las licencias de Project Gutenberg usando `.find()` para ubicar el inicio y fin del libro.
* Se seleccionan solo los primeros 1,100,000 caracteres correspondientes a la Primera Parte.

**Funciones y parámetros:**
* `keras.utils.get_file(fname, origin)`: Descarga el archivo desde la URL provista y lo guarda en caché.
* `open(..., encoding="utf-8-sig")`: Abre el archivo de texto soportando caracteres especiales del español.

In [3]:
import keras
import tensorflow as tf
from keras import layers
import numpy as np

# 1. Descargar Don Quijote (ID 2000) desde Project Gutenberg
quijote_url = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
filename = keras.utils.get_file(
    fname="don_quijote.txt",
    origin=quijote_url,
)

# 2. Leer con codificación UTF-8-sig (elimina el BOM inicial y preserva tildes y ñ)
with open(filename, "r", encoding="utf-8-sig") as f:
    raw_text = f.read()

# 3. Recortar las licencias en inglés de Project Gutenberg
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

start_idx = raw_text.find("\n", raw_text.find(start_marker)) + 1 if start_marker in raw_text else 0
end_idx = raw_text.find(end_marker) if end_marker in raw_text else len(raw_text)

quijote_clean = raw_text[start_idx:end_idx].strip()

# 4. Tomar exactamente los primeros 1,100,000 caracteres (Primera Parte del Quijote)
quijote = quijote_clean[:1_100_000]

print(f"Longitud del corpus seleccionado: {len(quijote):,} caracteres")
print("\n--- Muestra de los primeros 250 caracteres ---")
print(quijote[:250])

2225803/2225803 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Longitud del corpus seleccionado: 1,100,000 caracteres

--- Muestra de los primeros 250 caracteres ---
El ingenioso hidalgo don Quijote de la Mancha



por Miguel de Cervantes Saavedra





El ingenioso hidalgo don Quijote de la Mancha


  
Tasa

  
Testimonio de las erratas

  
El Rey

  
Al Duque de Béjar

  
Prólogo

  
Al libro de don Quijote de l


### 1.2 Creación de Secuencias y Tokenización
Al igual que con Shakespeare, dividimos el texto en secuencias y vectorizamos los caracteres.

**Cambios respecto al ejemplo original:**
* El tamaño del vocabulario será diferente ya que incluimos caracteres del español (tildes, ñ).
* Se muestra información del tamaño del vocabulario de manera explícita.

**Funciones y parámetros:**
* `split_input`: Generador que corta el texto en fragmentos de `sequence_length=100`.
* `tf.data.Dataset.from_tensor_slices`: Crea un dataset altamente optimizado a partir de las listas de entrada (features) y etiquetas (labels).
* `layers.TextVectorization`: Capa para convertir texto a IDs. `standardize=None` evita que se pierdan mayúsculas y signos de puntuación. `split="character"` tokeniza el texto letra por letra.

In [4]:
sequence_length = 100

def split_input(input_text, sequence_length):
    for i in range(0, len(input_text), sequence_length):
        yield input_text[i : i + sequence_length]

# Desplazamiento de 1 carácter entre entrada (features) y objetivo (labels)
features = list(split_input(quijote[:-1], sequence_length))
labels = list(split_input(quijote[1:], sequence_length))
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

# Verificación rápida del desplazamiento
x_sample, y_sample = next(dataset.as_numpy_iterator())
print("Entrada (x):", x_sample.decode("utf-8")[:50])
print("Etiqueta(y):", y_sample.decode("utf-8")[:50])

# Tokenizador a nivel de caracteres sin estandarización (conserva mayúsculas y puntuación)
tokenizer = layers.TextVectorization(
    standardize=None,
    split="character",
    output_sequence_length=sequence_length,
)
tokenizer.adapt(dataset.map(lambda text, labels: text))

vocabulary_size = tokenizer.vocabulary_size()
print(f"\nTamaño del vocabulario (caracteres únicos + padding + UNK): {vocabulary_size}")

# Mapear texto a IDs enteros y empaquetar en lotes de 64
dataset = dataset.map(
    lambda features, labels: (tokenizer(features), tokenizer(labels)),
    num_parallel_calls=8,
)
training_data = dataset.shuffle(10_000).batch(64).cache()

Entrada (x): El ingenioso hidalgo don Quijote de la Mancha



p
Etiqueta(y): l ingenioso hidalgo don Quijote de la Mancha



po

Tamaño del vocabulario (caracteres únicos + padding + UNK): 91


### 1.3 Construcción y Entrenamiento del Modelo de Lenguaje
Construimos un modelo de red neuronal recurrente (RNN) basado en celdas GRU.

**Cambios respecto al ejemplo original:**
* Se aumentó el número de épocas de entrenamiento a 10 para intentar que el modelo aprenda mejor la compleja estructura del español antiguo de Cervantes.
* Se guarda el objeto `history` devuelto por `model.fit`, lo cual es útil si posteriormente queremos graficar las curvas de pérdida.

**Funciones y parámetros:**
* `layers.Embedding`: Transforma los IDs numéricos de los caracteres en vectores densos, permitiendo que la red aprenda relaciones espaciales.
* `layers.GRU`: Capa recurrente que mantiene un estado interno (memoria) de lo que ha leído previamente.
* `model.compile`: Prepara el modelo. Usa `adam` como optimizador y `sparse_categorical_crossentropy` como función de pérdida, ideal para clasificación de clases mutuamente excluyentes (un carácter contra el resto).

In [5]:
embedding_dim = 256
hidden_dim = 1024

inputs = layers.Input(shape=(sequence_length,), dtype="int", name="token_ids")
x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x = layers.GRU(hidden_dim, return_sequences=True)(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary(line_length=80)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)

# Guardamos el objeto history para poder graficar las curvas en el reporte
history = model.fit(training_data, epochs=10)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)            │ (None, 100)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ embedding (Embedding)             │ (None, 100, 256)         │        23,296 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ gru (GRU)                         │ (None, 100, 1024)        │     3,938,304 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout (Dropout)                 │ (None, 100, 1024)        │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense (Dense)                     │ (None, 100, 91)          │        93,275 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

 Total params: 4,054,875 (15.47 MB)

 Trainable params: 4,054,875 (15.47 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 19s 55ms/step - loss: 2.3602 - sparse_categorical_accuracy: 0.3262
Epoch 2/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 56ms/step - loss: 1.7730 - sparse_categorical_accuracy: 0.4589
Epoch 3/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 57ms/step - loss: 1.5442 - sparse_categorical_accuracy: 0.5248
Epoch 4/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 59ms/step - loss: 1.4196 - sparse_categorical_accuracy: 0.5583
Epoch 5/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 61ms/step - loss: 1.3437 - sparse_categorical_accuracy: 0.5783
Epoch 6/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 61ms/step - loss: 1.2870 - sparse_categorical_accuracy: 0.5938
Epoch 7/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 61ms/step - loss: 1.2399 - sparse_categorical_accuracy: 0.6065
Epoch 8/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 61ms/step - loss: 1.1975 - sparse_categorical_accuracy: 0.6188
Epoch 9/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 61ms/step - loss: 1.1571 - sparse_categorical_accuracy: 0.6301
Epoch 10/10
172/172 ━━━━━━━━━━━━━━━━━

### 1.4 Generación de Texto con Muestreo de Temperatura
Creamos un modelo de inferencia para generar texto carácter por carácter a partir de una semilla inicial (prompt).

**Cambios respecto al ejemplo original:**
* Se introduce el concepto de **Temperatura (Temperature Sampling)** en lugar de usar siempre `np.argmax` (Greedy Decoding). Esto permite controlar la aleatoriedad del modelo.
* Una temperatura baja (ej. 0.0) hace al modelo predecible, mientras que una temperatura alta (> 0.7) lo hace más creativo pero propenso a errores gramaticales o inventar palabras.

**Funciones y parámetros:**
* `sample_with_temperature`: Función personalizada que altera las probabilidades de predicción antes de seleccionar el siguiente carácter usando `np.random.multinomial`.
* `generation_model.set_weights`: Transfiere los pesos aprendidos durante el entrenamiento al modelo generativo paso a paso.

In [6]:
# 1. Instanciar el modelo de generación paso a paso (1 carácter a la vez)
inputs_gen = keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state = keras.Input(shape=(hidden_dim,), name="state")

x_gen = layers.Embedding(vocabulary_size, embedding_dim)(inputs_gen)
x_gen, output_state = layers.GRU(hidden_dim, return_state=True)(
    x_gen, initial_state=input_state
)
outputs_gen = layers.Dense(vocabulary_size, activation="softmax")(x_gen)

generation_model = keras.Model(
    inputs=(inputs_gen, input_state),
    outputs=(outputs_gen, output_state),
)
# Copiar los pesos aprendidos por el modelo de entrenamiento
generation_model.set_weights(model.get_weights())

# 2. Diccionarios de mapeo Caracter <-> ID
tokens = tokenizer.get_vocabulary()
char_to_id = dict(zip(tokens, range(vocabulary_size)))
id_to_char = dict(zip(range(vocabulary_size), tokens))

# 3. Función de muestreo con Temperatura + función generadora
def sample_with_temperature(preds, temperature=0.6):
    # preds tiene forma (1, 1, vocabulary_size) o (1, vocabulary_size)
    preds = np.squeeze(np.asarray(preds, dtype="float64"))
    if temperature <= 0:  # Modo Greedy (argmax puro como en el notebook original)
        return int(np.argmax(preds))
    preds = np.log(preds + 1e-10) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return int(np.argmax(probas))

def generate_quijote_text(prompt, max_length=300, temperature=0.6):
    # Filtrar caracteres que por alguna razón no estén en el vocabulario (fallback a espacio)
    input_ids = [char_to_id.get(c, char_to_id.get(" ", 1)) for c in prompt]
    state = keras.ops.zeros(shape=(1, hidden_dim))

    # Fase de "Warm-up": alimentar el prompt para construir el estado oculto inicial
    for token_id in input_ids:
        inputs_step = keras.ops.expand_dims([token_id], axis=0)
        predictions, state = generation_model.predict((inputs_step, state), verbose=0)

    # Fase de Generación autorregresiva
    generated_ids = []
    for _ in range(max_length):
        next_char = sample_with_temperature(predictions, temperature=temperature)
        generated_ids.append(next_char)
        inputs_step = keras.ops.expand_dims([next_char], axis=0)
        predictions, state = generation_model.predict((inputs_step, state), verbose=0)

    output_text = "".join([id_to_char[token_id] for token_id in generated_ids])
    return prompt + output_text

# 4. PRUEBA DE FUNCIONAMIENTO
prompt_prueba = "En un lugar de la Mancha, de cuyo nombre "

print("=== PRUEBA 1: GREEDY DECODING (Original del cuaderno, T=0) ===")
print(generate_quijote_text(prompt_prueba, max_length=250, temperature=0.0))

print("\n=== PRUEBA 2: TEMPERATURE SAMPLING (Conservador, T=0.5) ===")
print(generate_quijote_text(prompt_prueba, max_length=250, temperature=0.5))

print("\n=== PRUEBA 3: TEMPERATURE SAMPLING (Creativo, T=0.8) ===")
print(generate_quijote_text(prompt_prueba, max_length=250, temperature=0.8))

=== PRUEBA 1: GREEDY DECODING (Original del cuaderno, T=0) ===
En un lugar de la Mancha, de cuyo nombre de la cabeza de la cabeza de la cabeza de la cabeza, y a los que las componen y los amigos de la cabeza del caso que le dejó en la venta donde no se le dejaba de ser alguna
pena que los de su amo se le pusiesen en la venta, que no era muy de provecho

=== PRUEBA 2: TEMPERATURE SAMPLING (Conservador, T=0.5) ===
En un lugar de la Mancha, de cuyo nombre que los de su hermano de la Mancha






A LA ARGAMASILLA ARADASILLA
ARAACIONTE DEL TOLONO

A LA ARGAMASILLA, ACADÉMICO LARASILLA
ACROBASIALO, ACADÉMICO DE LA ARGAMASILLALA
ALAACIONTE
EL DELOBRONTTO

Pero de la Santa Hermandad que su amo se supa, sin

=== PRUEBA 3: TEMPERATURE SAMPLING (Creativo, T=0.8) ===
En un lugar de la Mancha, de cuyo nombre de la discurso de
las heridas que le hacían, hecho en su gusto, que son andas del castillo de Endiablada y tengo, que no dije la del
amante, y que del dicho tiempo habló cuando llegando y arram

## Punto 2

### 2.1 Carga y Limpieza del Dataset (Español a Inglés)
Para el Punto 2, entrenamos un modelo Seq2Seq para traducción utilizando un archivo CSV que contiene pares de oraciones.

**Cambios respecto al ejemplo original:**
* En lugar de usar `keras.utils.get_file` para un archivo de texto simple, leemos un archivo estructurado local usando `pandas`.
* Filtramos las oraciones basándonos en su longitud (entre 2 y 25 palabras) para asegurar que encajen en nuestra variable `sequence_length`.
* Hacemos un muestreo aleatorio de 10,000 pares para limitar el tiempo de entrenamiento computacional.
* Agregamos los tokens `[start]` y `[end]` a las oraciones en inglés (el idioma destino).

**Funciones y parámetros:**
* `pd.read_csv`: Carga el archivo en un DataFrame de Pandas.
* `re.sub`: Usa expresiones regulares para limpiar corchetes literales indeseados que podrían confundir al tokenizador.
* `random.shuffle`: Mezcla los datos para evitar sesgos antes de partirlos en train, val y test.

In [8]:
import pandas as pd
import random
import re

# Cambia el nombre si tu archivo en la raíz tiene guiones bajos o espacios distintos
csv_path = "Combined Data es-ES.csv"

# 1. Cargar solo las columnas necesarias y eliminar filas vacías (nulos)
df = pd.read_csv(csv_path, usecols=["statement", "statement_es_es"]).dropna()
print(f"Total de registros válidos en el CSV original: {len(df):,}")

# 2. Limpiar corchetes literales (como el ':([].' de la fila 8) para no romper [start] y [end]
df["statement_es_es"] = df["statement_es_es"].astype(str).apply(lambda s: re.sub(r"[\[\]]", "", s).strip())
df["statement"] = df["statement"].astype(str).apply(lambda s: re.sub(r"[\[\]]", "", s).strip())

# 3. Filtrar oraciones demasiado largas (máximo 25 palabras) o vacías para que quepan en la ventana de la GRU
mask_longitud = (
    (df["statement_es_es"].str.split().str.len().between(2, 25)) &
    (df["statement"].str.split().str.len().between(2, 25))
)
df_filtrado = df[mask_longitud]
print(f"Registros con longitud óptima (2 a 25 palabras): {len(df_filtrado):,}")

# 4. Muestreo aleatorio de exactamente 10,000 ejemplos (semilla fija para reproducibilidad)
n_samples = min(10_000, len(df_filtrado))
df_sample = df_filtrado.sample(n=n_samples, random_state=42).reset_index(drop=True)

# 5. Construir los pares (ORIGEN: Español -> DESTINO: Inglés con [start] y [end])
text_pairs = []
for _, row in df_sample.iterrows():
    spanish = row["statement_es_es"]
    english = "[start] " + row["statement"] + " [end]"
    text_pairs.append((spanish, english))

#Liberar memoria RAM del DataFrame pesado
del df, df_filtrado

# 6. Partición 70% Entrenamiento (7,000) / 15% Validación (1,500) / 15% Prueba (1,500)
random.seed(42)
random.shuffle(text_pairs)

val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples

train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

print(f"\nTotal seleccionado : {len(text_pairs):,}")
print(f" Train pairs (70%) : {len(train_pairs):,}")
print(f" Val pairs   (15%) : {len(val_pairs):,}")
print(f" Test pairs  (15%) : {len(test_pairs):,}")
print("\nEjemplo de par (Español -> Inglés):")
print(random.choice(train_pairs))

Total de registros válidos en el CSV original: 52,680
Registros con longitud óptima (2 a 25 palabras): 17,389

Total seleccionado : 10,000
 Train pairs (70%) : 7,000
 Val pairs   (15%) : 1,500
 Test pairs  (15%) : 1,500

Ejemplo de par (Español -> Inglés):
('El amor entra en el corazón de una persona sin motivo. Y controla cada vez más el corazón de alguien.', "[start] love enters a person's heart for no reason. And will control someone's heart more and more [end]")


### 2.2 Tokenización con Estandarización Personalizada
Configuramos dos tokenizadores (uno para el Encoder y otro para el Decoder).

**Cambios respecto al ejemplo original:**
* Se añade un filtrado de caracteres especiales como "¡" y "¿", propios del español.
* Se utilizan vocabularios limitados a 12,000 palabras y longitudes de secuencia fijas de 25.
* El tokenizador en inglés tiene un tamaño de salida de `sequence_length + 1` para permitir el desfase temporal durante el entrenamiento (Teacher Forcing).

**Funciones y parámetros:**
* `custom_standardization`: Función que pasa el texto a minúsculas y elimina los signos de puntuación definidos usando `tf.strings.regex_replace`.
* `layers.TextVectorization`: Mapea palabras completas (en lugar de caracteres) a IDs enteros.

In [9]:
import string
import tensorflow as tf
from keras import layers

strip_chars = string.punctuation + "¿¡"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

vocab_size = 12000      # Óptimo para un subconjunto de 10,000 oraciones
sequence_length = 25    # Cubre el 100% de nuestro filtro de 25 palabras

# Encoder Tokenizer: Español (longitud = sequence_length)
spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
    standardize=custom_standardization,
)

# Decoder Tokenizer: Inglés (longitud = sequence_length + 1 para Teacher Forcing)
english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)

train_spanish_texts = [pair[0] for pair in train_pairs]
train_english_texts = [pair[1] for pair in train_pairs]

spanish_tokenizer.adapt(train_spanish_texts)
english_tokenizer.adapt(train_english_texts)

print(f"Vocabulario Español adaptado: {spanish_tokenizer.vocabulary_size()} tokens")
print(f"Vocabulario Inglés adaptado : {english_tokenizer.vocabulary_size()} tokens")

Vocabulario Español adaptado: 12000 tokens
Vocabulario Inglés adaptado : 11271 tokens


### 2.3 Empaquetado en TensorFlow Datasets
Creamos los datasets altamente optimizados para alimentar al modelo durante el entrenamiento.

**Cambios respecto al ejemplo original:**
* La tokenización se realiza previamente en memoria (`spanish_tokenizer(...)`) en lugar de usar un `.map()` dentro del pipeline de TensorFlow. Esto suele ser más rápido y evita conflictos cuando se utilizan backends como JAX.

**Funciones y parámetros:**
* `format_dataset`: Construye la estructura requerida. Asigna el español completo a la entrada del Encoder, y el inglés (sin el último token) a la entrada del Decoder. El objetivo (label) es el inglés desplazado un token hacia adelante.
* `sample_weights = labels != 0`: Genera una máscara booleana para ignorar el padding (los ceros) al calcular la pérdida, evitando que el modelo dedique esfuerzo en predecir espacios en blanco.

In [10]:
batch_size = 64

# Pre-tokenización en memoria
train_spa_tokenized = spanish_tokenizer(train_spanish_texts)
train_eng_tokenized = english_tokenizer(train_english_texts)

val_spa_texts = [pair[0] for pair in val_pairs]
val_eng_texts = [pair[1] for pair in val_pairs]
val_spa_tokenized = spanish_tokenizer(val_spa_texts)
val_eng_tokenized = english_tokenizer(val_eng_texts)

def format_dataset(spa_tokens, eng_tokens):
    features = {
        "spanish": spa_tokens,          # Entrada al Encoder
        "english": eng_tokens[:, :-1],  # Entrada al Decoder (con [start], sin último token)
    }
    labels = eng_tokens[:, 1:]          # Objetivo del Decoder (sin [start], con [end])
    sample_weights = labels != 0        # Máscara para ignorar el padding (0)
    return features, labels, sample_weights

def make_dataset(spa_tokens, eng_tokens):
    dataset = tf.data.Dataset.from_tensor_slices((spa_tokens, eng_tokens))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_spa_tokenized, train_eng_tokenized)
val_ds = make_dataset(val_spa_tokenized, val_eng_tokenized)

# Verificación de dimensiones del lote
inputs, targets, sample_weights = next(iter(train_ds))
print("Shape input 'spanish':", inputs["spanish"].shape)
print("Shape input 'english':", inputs["english"].shape)
print("Shape targets        :", targets.shape)

Shape input 'spanish': (64, 25)
Shape input 'english': (64, 25)
Shape targets        : (64, 25)


### 2.4 Modelo Seq2Seq (Encoder-Decoder) y Entrenamiento
Construimos la arquitectura de traducción que consta de dos partes interconectadas.

**Cambios respecto al ejemplo original:**
* Se aumentó el número de épocas a 15 para tratar de mejorar los resultados en este dataset de oraciones más diversas.

**Funciones y parámetros:**
* `layers.Bidirectional`: Hace que la GRU del Encoder lea la oración de izquierda a derecha y viceversa, permitiendo capturar un mejor contexto general.
* `initial_state=encoder_output`: Este es el puente del modelo Seq2Seq. Pasa la representación matemática que el Encoder generó (resumen del texto en español) como el estado inicial del Decoder para comenzar a generar en inglés.

In [11]:
import keras

embed_dim = 256
hidden_dim = 1024

# 1. ENCODER (Recibe Español)
source = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, embed_dim)(source)
encoder_rnn = layers.Bidirectional(layers.GRU(hidden_dim), merge_mode="sum")
encoder_output = encoder_rnn(x)

# 2. DECODER (Recibe Inglés desplazado + estado del Encoder)
target = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, embed_dim)(target)
decoder_rnn = layers.GRU(hidden_dim, return_sequences=True)
x = decoder_rnn(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)

seq2seq_rnn = keras.Model([source, target], target_predictions)
seq2seq_rnn.summary(line_length=80)

seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)

history_seq2seq = seq2seq_rnn.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_2           │ (None, None, 256) │   3,072,000 │ spanish[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_3           │ (None, None, 256) │   3,072,000 │ english[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ bidirectional         │ (None, 1024)      │   7,876,608 │ embedding_2[0][0]  │
│ (Bidirectional)       │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ gru_3 (GRU)           │ (None, None,      │   3,938,304 │ embedding_3[0][0], │
│                       │ 1024)             │             │ bidirectional[0][… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_1 (Dropout)   │ (None, None,      │           0 │ gru_3[0][0]        │
│                       │ 1024)             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense_2 (Dense)       │ (None, None,      │  12,300,000 │ dropout_1[0][0]    │
│                       │ 12000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 30,258,912 (115.43 MB)

 Trainable params: 30,258,912 (115.43 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 23s 169ms/step - accuracy: 0.1075 - loss: 3.2782 - val_accuracy: 0.1422 - val_loss: 2.9670
Epoch 2/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 19s 170ms/step - accuracy: 0.1502 - loss: 2.8756 - val_accuracy: 0.1617 - val_loss: 2.8360
Epoch 3/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 19s 174ms/step - accuracy: 0.1735 - loss: 2.6772 - val_accuracy: 0.1654 - val_loss: 2.8174
Epoch 4/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 20s 178ms/step - accuracy: 0.1917 - loss: 2.5045 - val_accuracy: 0.1823 - val_loss: 2.8002
Epoch 5/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 20s 180ms/step - accuracy: 0.2085 - loss: 2.3311 - val_accuracy: 0.1811 - val_loss: 2.8164
Epoch 6/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 20s 183ms/step - accuracy: 0.2244 - loss: 2.1593 - val_accuracy: 0.1950 - val_loss: 2.8289
Epoch 7/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 20s 181ms/step - accuracy: 0.2506 - loss: 1.9807 - val_accuracy: 0.1995 - val_loss: 2.8602
Epoch 8/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 20s 180ms/step - accuracy: 0.2914 - loss: 1

### 2.5 Inferencia y Evaluación con Métricas Avanzadas
Traducimos nuevas oraciones y evaluamos objetivamente la calidad del modelo.

**Cambios respecto al ejemplo original:**
* En lugar de solo imprimir algunas traducciones, implementamos métricas estándar de procesamiento de lenguaje natural (NLP): **BLEU** (evalúa la precisión de n-gramas) y **ROUGE** (orientado al recall), además de un **F1-Score** a nivel de tokens.
* Se utiliza una función de predicción vectorizada por lotes (`translate_batch_es_to_en`) que traduce múltiples oraciones a la vez de forma mucho más eficiente que un bucle iterativo oración por oración.

**Funciones y parámetros:**
* `corpus_bleu`: Calcula el puntaje BLEU comparando las oraciones predichas con las referencias reales.
* `rouge_scorer`: Calcula la superposición de unigramas (ROUGE-1), bigramas (ROUGE-2) y la subsecuencia común más larga (ROUGE-L) para medir similitud de significado.

In [12]:
!pip install nltk rouge-score -q

import numpy as np
from collections import Counter
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer

eng_vocab = english_tokenizer.get_vocabulary()
eng_index_lookup = dict(zip(range(len(eng_vocab)), eng_vocab))

# 1. Función de traducción vectorizada por lotes (traduce N oraciones en paralelo)
def translate_batch_es_to_en(spanish_sentences):
    batch_len = len(spanish_sentences)
    tokenized_inputs = spanish_tokenizer(spanish_sentences)
    decoded_sentences = ["[start]"] * batch_len
    finished = [False] * batch_len

    for i in range(sequence_length):
        tokenized_targets = english_tokenizer(decoded_sentences)[:, :-1]
        preds = seq2seq_rnn.predict(
            {"spanish": tokenized_inputs, "english": tokenized_targets},
            batch_size=64,
            verbose=0,
        )
        sampled_indices = np.argmax(preds[:, i, :], axis=-1)
        for b in range(batch_len):
            if not finished[b]:
                token = eng_index_lookup[sampled_indices[b]]
                if token == "[end]" or token == "":
                    finished[b] = True
                else:
                    decoded_sentences[b] += " " + token
        if all(finished):
            break

    return [s.replace("[start]", "").strip() for s in decoded_sentences]

# 2. Función para F1-Score a nivel de tokens (Bag-of-Tokens Overlap)
def compute_token_f1(ref_tokens, pred_tokens):
    if not ref_tokens or not pred_tokens:
        return 0.0
    common = Counter(ref_tokens) & Counter(pred_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)
    return (2 * precision * recall) / (precision + recall)

# 3. Evaluar sobre 256 pares del conjunto de prueba (test_pairs)
num_eval = min(256, len(test_pairs))
eval_subset = test_pairs[:num_eval]

spa_test_inputs = [pair[0] for pair in eval_subset]
# Pasamos las referencias reales por custom_standardization para evaluar en igualdad de condiciones (sin signos ni mayúsculas)
eng_test_refs = [
    custom_standardization(pair[1].replace("[start]", "").replace("[end]", "")).numpy().decode("utf-8").strip()
    for pair in eval_subset
]

print(f"Generando traducciones para {num_eval} oraciones de prueba...")
eng_predictions = translate_batch_es_to_en(spa_test_inputs)

# 4. Cálculo de BLEU, ROUGE y F1
references_bleu = []
candidates_bleu = []
rouge1_list, rouge2_list, rougel_list, f1_list = [], [], [], []

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
smoothie = SmoothingFunction().method4

for ref_str, pred_str in zip(eng_test_refs, eng_predictions):
    ref_tokens = ref_str.split()
    pred_tokens = pred_str.split()

    references_bleu.append([ref_tokens])
    candidates_bleu.append(pred_tokens)

    r_scores = scorer.score(ref_str, pred_str)
    rouge1_list.append(r_scores["rouge1"].fmeasure)
    rouge2_list.append(r_scores["rouge2"].fmeasure)
    rougel_list.append(r_scores["rougeL"].fmeasure)

    f1_list.append(compute_token_f1(ref_tokens, pred_tokens))

bleu_1 = corpus_bleu(references_bleu, candidates_bleu, weights=(1.0, 0, 0, 0), smoothing_function=smoothie)
bleu_4 = corpus_bleu(references_bleu, candidates_bleu, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

print("\n" + "="*45)
print("   RESULTADOS DE EVALUACIÓN SEQ2SEQ (ES -> EN)")
print("="*45)
print(f"BLEU-1 (Unigramas)      : {bleu_1:.4f}")
print(f"BLEU-4 (Estándar)       : {bleu_4:.4f}")
print(f"ROUGE-1 (F1 promedio)   : {np.mean(rouge1_list):.4f}")
print(f"ROUGE-2 (F1 promedio)   : {np.mean(rouge2_list):.4f}")
print(f"ROUGE-L (F1 promedio)   : {np.mean(rougel_list):.4f}")
print(f"Token F1-Score promedio : {np.mean(f1_list):.4f}")
print("="*45)

# 5. Mostrar 5 ejemplos cualitativos
print("\n--- EJEMPLOS DE TRADUCCIÓN EN TEST ---")
for i in range(5):
    print(f"\n[{i+1}] Español (Entrada)   : {spa_test_inputs[i]}")
    print(f"    Inglés (Referencia) : {eng_test_refs[i]}")
    print(f"    Inglés (Predicción) : {eng_predictions[i]}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Generando traducciones para 256 oraciones de prueba...

   RESULTADOS DE EVALUACIÓN SEQ2SEQ (ES -> EN)
BLEU-1 (Unigramas)      : 0.1423
BLEU-4 (Estándar)       : 0.0182
ROUGE-1 (F1 promedio)   : 0.1652
ROUGE-2 (F1 promedio)   : 0.0482
ROUGE-L (F1 promedio)   : 0.1572
Token F1-Score promedio : 0.1600

--- EJEMPLOS DE TRADUCCIÓN EN TEST ---

[1] Español (Entrada)   : no puedo soportar esto más, mejor terminar con todo lo siento que la vida
    Inglés (Referencia) : i cant take this anymore might as well end this sorry as life
    Inglés (Predicción) : i am not feeling better than my thoughts are so many plaques that i want to sleep

[2] Español (Entrada)   : Quiero una jeringa
    Inglés (Referencia) : want a syringe
    Inglés (Predicción) : want to be accompanied

[3] Español (Entrada)   : mejor ver jumanji 2 o los incredibles 2?
    Inglés (Referencia)

### Conclusiones a partir de los resultados
Analizando las salidas y métricas de ambos modelos, concluimos lo siguiente:

1. **Punto 1 (Modelo Generativo de Lenguaje - Don Quijote):**
   * El modelo logra interiorizar la sintaxis básica del idioma español y la estructura visual de una novela (saltos de línea, diálogos).
   * El uso de la técnica de **Temperatura (Sampling)** es fundamental. Con decodificación codiciosa (Greedy, T=0), el modelo pierde el hilo y entra en bucles infinitos ("de la cabeza de la cabeza"). A medida que subimos la temperatura, se vuelve más fluido, aunque con T=0.8 comienza a inventar palabras sin sentido ("arramana"). Una temperatura moderada proporciona el mejor equilibrio.

2. **Punto 2 (Traducción Seq2Seq ES -> EN):**
   * Las métricas de evaluación cuantitativa son bastante bajas (BLEU-1 cercano a 0.14 y BLEU-4 menor a 0.02).
   * Al observar las predicciones cualitativas, vemos que el modelo identifica algunas palabras clave sueltas (ej. "sleep", "want"), pero fracasa rotundamente en mantener la coherencia semántica y el orden gramatical en oraciones medianas o largas.
   * **Causas principales:**
     - El conjunto de datos de entrenamiento (10,000 muestras) es insuficiente para aprender a traducir de manera generalizada.
     - Los modelos basados puramente en arquitecturas RNN/GRU tradicionales sufren el problema del "cuello de botella", ya que deben comprimir toda la oración en un solo vector. Para mejorar esto, la literatura moderna utiliza el **mecanismo de Atención** o directamente arquitecturas basadas en **Transformers**.